tiny model 96x96x3 


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set legacy Keras mode BEFORE any TensorFlow/Keras imports
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras  # Use tf_keras for TFMOT compatibility
from tf_keras import layers
from tf_keras.preprocessing.image import ImageDataGenerator
import tensorflow_model_optimization as tfmot
import tensorflow.lite as tflite

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("CPUs:", tf.config.list_physical_devices("CPU"))



TF version: 2.16.1
GPUs: []
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
DATASET_PATH = "D:/waste_project/waste-classification-system-II/resized_dataset"

img_size = (96, 96)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="rgb"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="rgb"
)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print("Classes:", class_names)
print("Number of classes:", num_classes)


Found 8407 files belonging to 9 classes.
Using 6726 files for training.
Found 8407 files belonging to 9 classes.
Using 1681 files for validation.
Classes: ['cardboard', 'e-waste', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
Number of classes: 9


In [3]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)


In [4]:
from tensorflow import keras
from tensorflow.keras import layers, models

num_classes = 9

def tiny_cnn_96x96_rgb():
    inputs = keras.Input(shape=(96, 96, 3))
    x = layers.Rescaling(1./255)(inputs)

    # Block 1
    x = layers.SeparableConv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Block 2
    x = layers.SeparableConv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Block 3
    x = layers.SeparableConv2D(96, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Block 4 (small but helpful)
    x = layers.SeparableConv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    # Tiny head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(48, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = tiny_cnn_96x96_rgb()

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 rescaling (Rescaling)       (None, 96, 96, 3)         0         
                                                                 
 separable_conv2d (Separabl  (None, 96, 96, 32)        155       
 eConv2D)                                                        
                                                                 
 batch_normalization (Batch  (None, 96, 96, 32)        128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 48, 48, 32)        0         
 D)                                                              
                                                            

In [5]:
es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[es]
)


Epoch 1/40


211/211 [==============================] - 80s 225ms/step - loss: 1.8204 - accuracy: 0.3402 - val_loss: 2.2262 - val_accuracy: 0.1095
Epoch 2/40
211/211 [==============================] - 45s 205ms/step - loss: 1.6282 - accuracy: 0.4205 - val_loss: 2.1562 - val_accuracy: 0.1957
Epoch 3/40
211/211 [==============================] - 46s 207ms/step - loss: 1.5477 - accuracy: 0.4505 - val_loss: 1.6464 - val_accuracy: 0.4503
Epoch 4/40
211/211 [==============================] - 46s 210ms/step - loss: 1.4796 - accuracy: 0.4737 - val_loss: 1.5393 - val_accuracy: 0.4557
Epoch 5/40
211/211 [==============================] - 46s 210ms/step - loss: 1.4352 - accuracy: 0.4917 - val_loss: 1.5144 - val_accuracy: 0.4777
Epoch 6/40
211/211 [==============================] - 47s 215ms/step - loss: 1.3863 - accuracy: 0.5085 - val_loss: 1.4183 - val_accuracy: 0.5122
Epoch 7/40
211/211 [==============================] - 47s 213ms/step - loss: 1.3448 - accuracy: 0.5229 - val_loss: 1.6679 - val_

In [6]:
import tensorflow_model_optimization as tfmot

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

def prune_dense(layer):
    if isinstance(layer, keras.layers.Dense):
        return prune_low_magnitude(
            layer,
            pruning_schedule=tfmot.sparsity.keras.PolynomialDecay(
                initial_sparsity=0.0,
                final_sparsity=0.5,
                begin_step=0,
                end_step=1000
            )
        )
    return layer

pruned_model = keras.models.clone_model(model, clone_function=prune_dense)

pruned_model.compile(
    optimizer=keras.optimizers.Adam(5e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
211/211 [==============================] - 54s 210ms/step - loss: 1.0811 - accuracy: 0.6201 - val_loss: 1.1430 - val_accuracy: 0.6002
Epoch 2/10
211/211 [==============================] - 46s 208ms/step - loss: 1.0785 - accuracy: 0.6228 - val_loss: 1.1687 - val_accuracy: 0.5919
Epoch 3/10
211/211 [==============================] - 45s 205ms/step - loss: 1.0907 - accuracy: 0.6194 - val_loss: 1.1243 - val_accuracy: 0.6211
Epoch 4/10
211/211 [==============================] - 49s 222ms/step - loss: 1.0730 - accuracy: 0.6262 - val_loss: 1.1427 - val_accuracy: 0.6044
Epoch 5/10
211/211 [==============================] - 46s 210ms/step - loss: 1.0681 - accuracy: 0.6240 - val_loss: 1.1784 - val_accuracy: 0.5889
Epoch 6/10
211/211 [==============================] - 46s 210ms/step - loss: 1.0585 - accuracy: 0.6271 - val_loss: 1.1103 - val_accuracy: 0.6169
Epoch 7/10
211/211 [==============================] - 46s 207ms/step - loss: 1.0534 - accuracy: 0.6331 - val_loss: 1.1056 - val_ac

In [7]:
model_for_export = tfmot.sparsity.keras.strip_pruning(pruned_model)


In [8]:
def representative_data_gen():
    for images, _ in train_ds.take(100):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("tiny_cnn_96x96_rgb_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("INT8 size (KB):", len(tflite_model) / 1024)


INFO:tensorflow:Assets written to: C:\Users\Asghar\AppData\Local\Temp\tmp643pa5q1\assets


INFO:tensorflow:Assets written to: C:\Users\Asghar\AppData\Local\Temp\tmp643pa5q1\assets
C:\Users\Asghar\anaconda3\envs\waste_env\lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 size (KB): 52.671875
